# 대청호 조류경보 예측 모델 학습 코드

이 노트북은 **모델 학습 코드만** 보기 좋게 정리한 파일입니다.

목표는 수질·수문·기상 데이터를 채수일 기준으로 통합하고, 도메인 지식 기반 파생변수를 만든 뒤 다음 채수일의 유해남조류 세포수와 관심 이상 여부를 예측하는 것입니다.

사용 모델은 설명력과 현실성을 고려해 `HistGradientBoosting`과 `RandomForest`를 사용합니다.

## 1. 모델링 개요

```text
수질·수문·기상 데이터
        ↓
채수일 기준 통합
        ↓
도메인 기반 파생변수 생성
- TSI proxy
- HRT / residence proxy
- net flow
- 강우·일조·풍속 rolling feature
- 회남 -> 추동 -> 문의 graph feature
        ↓
Gradient Boosting 학습
        ↓
세포수 예측 + 관심 이상 확률 예측
```

주의: 현재 데이터에는 TP가 없으므로 TSI는 완전한 Carlson TSI가 아니라 `Chl-a`와 `투명도`를 활용한 **TSI proxy**입니다.

In [ ]:
# 필요 패키지
import os
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. 데이터 경로 설정

데이터는 레포에 올리지 않습니다. 아래 경로만 본인 로컬 환경에 맞게 수정하면 됩니다.

기본값은 이 레포와 `dam` 폴더가 같은 위치에 있다고 가정합니다.

In [ ]:
DATA_ROOT = Path(os.getenv("DATA_ROOT", "../dam/data/processed"))

QUALITY_CSV = Path(os.getenv(
    "QUALITY_CSV",
    DATA_ROOT / "kwater_designated/daecheong_algae_monitoring_daily.csv",
))
DAM_CSV = Path(os.getenv(
    "DAM_CSV",
    DATA_ROOT / "kwater_designated/daecheong_dam_operations_daily.csv",
))
KMA_CSV = Path(os.getenv(
    "KMA_CSV",
    DATA_ROOT / "kma/OBS_ASOS_TIM_20260424114201.csv",
))
GEUM_DAM_CSV = Path(os.getenv(
    "GEUM_DAM_CSV",
    DATA_ROOT / "geumriver_history/dmList_2016-01-01_2026-04-25_0000.csv",
))

OUTPUT_DIR = Path(os.getenv("MODEL_OUTPUT_DIR", "outputs/model_pipeline"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)

paths = {
    "quality": QUALITY_CSV,
    "dam": DAM_CSV,
    "kma": KMA_CSV,
    "geum_dam": GEUM_DAM_CSV,
    "output": OUTPUT_DIR,
}
paths

In [ ]:
# 필수 데이터 확인
required = [QUALITY_CSV, DAM_CSV]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("필수 데이터가 없습니다. 경로를 확인하세요: " + ", ".join(missing))

## 3. 기본 설정과 유틸 함수

In [ ]:
TARGET_COL = "유해남조류 세포수 (cells/㎖) (1+2+3+4)"
SITE_ORDER = {"회남": 0, "추동": 1, "문의": 2}
SITE_TO_STATION = {"회남": "보은", "추동": "대전", "문의": "청주"}
UPSTREAM = {"회남": None, "추동": "회남", "문의": "추동"}
RANDOM_STATE = 42


def to_num(series):
    return pd.to_numeric(
        series.replace({"-": np.nan, "&nbsp;": np.nan, "정량한계미만": np.nan}),
        errors="coerce",
    )


def safe_div(a, b):
    return a / (b + 1e-6)


def log1p10(values):
    return np.log10(np.maximum(values, 0) + 1)


def positive_ln(values):
    values = pd.to_numeric(values, errors="coerce")
    return np.log(values.where(values > 0))

## 4. 데이터 로드

In [ ]:
def load_quality(path):
    q = pd.read_csv(path, low_memory=False)
    q["sample_date"] = pd.to_datetime(q["조사일"], errors="coerce")
    q["site"] = q["채수위치"].astype(str)

    numeric_cols = [
        TARGET_COL,
        "(1) Microcystis",
        "(2) Anabaena",
        "(3) Oscillatoria",
        "(4) Aphanizomenon",
        "수온(℃)",
        "pH",
        "DO(㎎/L)",
        "투명도",
        "탁도",
        "Chl-a (㎎/㎥)",
        "일조시간 합계(hr)",
        "일강수량(mm)",
    ]
    for col in numeric_cols:
        if col in q.columns:
            q[col] = to_num(q[col])
    return q.sort_values(["site", "sample_date"]).reset_index(drop=True)


def load_dam(path):
    dam = pd.read_csv(path, low_memory=False)
    dam["date"] = pd.to_datetime(dam["일시"], errors="coerce")
    rename = {
        "수위(EL.m)": "dam_level",
        "저수량(백만㎥)": "storage_volume",
        "저수율(%)": "storage_rate",
        "강우량(mm)": "rain",
        "유입량(㎥/s)": "inflow",
        "총방류량(㎥/s)": "outflow",
    }
    dam = dam.rename(columns=rename)
    for col in rename.values():
        dam[col] = to_num(dam[col])
    return dam[["date", *rename.values()]].sort_values("date")


def load_geum_dam(path):
    if not path.exists():
        return None
    geum = pd.read_csv(path, low_memory=False)
    if "OBSNM" in geum.columns:
        geum = geum[geum["OBSNM"].astype(str).eq("대청댐")].copy()
    geum["date"] = pd.to_datetime(geum["snapshot_datetime"].astype(str), format="%Y%m%d%H%M", errors="coerce")
    rename = {
        "SWL": "geum_level",
        "SFW": "geum_storage_volume",
        "VOL": "geum_storage_rate",
        "INF": "geum_inflow",
        "TOTOTF": "geum_outflow",
    }
    geum = geum.rename(columns=rename)
    keep = ["date", *[v for v in rename.values() if v in geum.columns]]
    for col in keep:
        if col != "date":
            geum[col] = to_num(geum[col])
    return geum[keep].sort_values("date")


def load_kma_daily(path):
    if not path.exists():
        return None
    kma = pd.read_csv(path, low_memory=False)
    kma["datetime"] = pd.to_datetime(kma["일시"], errors="coerce")
    kma["date"] = kma["datetime"].dt.floor("D")

    numeric = ["기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "일조(hr)", "일사(MJ/m2)", "전운량(10분위)"]
    for col in numeric:
        if col in kma.columns:
            kma[col] = to_num(kma[col])

    return (
        kma.groupby(["지점명", "date"], as_index=False)
        .agg(
            avg_temp=("기온(°C)", "mean"),
            max_temp=("기온(°C)", "max"),
            daily_rain=("강수량(mm)", "sum"),
            avg_wind=("풍속(m/s)", "mean"),
            sunshine=("일조(hr)", "sum"),
            solar_rad=("일사(MJ/m2)", "sum"),
            cloud_cover=("전운량(10분위)", "mean"),
        )
        .sort_values(["지점명", "date"])
    )

In [ ]:
quality_raw = load_quality(QUALITY_CSV)
dam_raw = load_dam(DAM_CSV)
kma_daily = load_kma_daily(KMA_CSV)
geum_dam = load_geum_dam(GEUM_DAM_CSV)

print("quality:", quality_raw.shape, quality_raw["sample_date"].min(), "~", quality_raw["sample_date"].max())
print("dam:", dam_raw.shape, dam_raw["date"].min(), "~", dam_raw["date"].max())
print("kma:", None if kma_daily is None else kma_daily.shape)
print("geum:", None if geum_dam is None else geum_dam.shape)

## 5. 파생 피처 생성

### 핵심 도메인 피처

| 피처 | 의미 |
| --- | --- |
| `net_flow` | 유입량 - 방류량 |
| `residence_proxy` | 저수량 / 방류량 |
| `level_change_3d`, `level_change_7d` | 최근 수위 변화 |
| `tsi_chla`, `tsi_transparency`, `tsi_proxy_mean` | Chl-a·투명도 기반 영양상태 proxy |
| `temp_light_growth_proxy` | 수온과 일조시간을 결합한 성장 조건 |
| `graph_decay_signal` | 회남 -> 추동 -> 문의 전파 신호 |

In [ ]:
def add_rolling_features(daily, prefix, windows=(3, 7, 14), sum_cols=(), mean_cols=()):
    daily = daily.copy().sort_values("date")
    out = daily[["date"]].copy()

    for col in sum_cols:
        for window in windows:
            out[f"{prefix}_{col}_{window}d_sum"] = daily[col].rolling(window, min_periods=1).sum()
        for lag in (1, 3, 7):
            out[f"{prefix}_{col}_lag{lag}"] = daily[col].shift(lag)

    for col in mean_cols:
        for window in windows:
            out[f"{prefix}_{col}_{window}d_mean"] = daily[col].rolling(window, min_periods=1).mean()
    return out


def build_hydro_features(dam, geum=None):
    dam = dam.copy()
    dam["net_flow"] = dam["inflow"] - dam["outflow"]
    dam["residence_proxy"] = safe_div(dam["storage_volume"], dam["outflow"])
    dam["nutrient_stagnation"] = safe_div(dam["rain"] * dam["inflow"], dam["outflow"] + 1)

    feat = add_rolling_features(
        dam,
        "hydro",
        sum_cols=("rain", "inflow", "outflow", "net_flow"),
        mean_cols=("dam_level", "storage_rate", "residence_proxy", "nutrient_stagnation"),
    )
    feat["level_change_3d"] = dam["dam_level"] - dam["dam_level"].shift(3)
    feat["level_change_7d"] = dam["dam_level"] - dam["dam_level"].shift(7)
    feat["moderate_rain_days_10_30_7d"] = dam["rain"].between(10, 30).rolling(7, min_periods=1).sum()
    feat["heavy_rain_80_7d"] = dam["rain"].rolling(7, min_periods=1).max().ge(80).astype(float)

    if geum is not None and not geum.empty:
        geum = geum.copy()
        geum["geum_net_flow"] = geum["geum_inflow"] - geum["geum_outflow"]
        geum["geum_residence_proxy"] = safe_div(geum["geum_storage_volume"], geum["geum_outflow"])
        geum_feat = add_rolling_features(
            geum,
            "geum",
            sum_cols=("geum_inflow", "geum_outflow", "geum_net_flow"),
            mean_cols=("geum_level", "geum_storage_rate", "geum_residence_proxy"),
        )
        feat = feat.merge(geum_feat, on="date", how="outer")

    return feat.rename(columns={"date": "sample_date"}).sort_values("sample_date")


def build_weather_features(samples, kma_daily):
    if kma_daily is None or kma_daily.empty:
        return samples[["site", "sample_date"]].drop_duplicates().copy()

    rows = []
    for site, station in SITE_TO_STATION.items():
        station_daily = kma_daily[kma_daily["지점명"].eq(station)].copy().sort_values("date")
        if station_daily.empty:
            continue
        roll = add_rolling_features(
            station_daily,
            "weather",
            sum_cols=("daily_rain", "sunshine", "solar_rad"),
            mean_cols=("avg_temp", "max_temp", "avg_wind", "cloud_cover"),
        )
        roll["site"] = site
        roll["hot_days_30c_7d"] = station_daily["max_temp"].ge(30).rolling(7, min_periods=1).sum()
        roll["low_wind_days_2ms_7d"] = station_daily["avg_wind"].le(2).rolling(7, min_periods=1).sum()
        roll["effective_light_7d"] = roll["weather_solar_rad_7d_sum"] * (1 - roll["weather_cloud_cover_7d_mean"] / 10)
        rows.append(roll.rename(columns={"date": "sample_date"}))

    if not rows:
        return samples[["site", "sample_date"]].drop_duplicates().copy()
    return pd.concat(rows, ignore_index=True)

In [ ]:
def add_quality_features(q):
    df = q.copy()
    rename = {
        "수온(℃)": "water_temp",
        "pH": "ph",
        "DO(㎎/L)": "do",
        "투명도": "transparency",
        "탁도": "turbidity",
        "Chl-a (㎎/㎥)": "chla",
        "일조시간 합계(hr)": "sunshine_obs",
        "일강수량(mm)": "rain_obs",
        "(1) Microcystis": "microcystis",
        "(2) Anabaena": "anabaena",
        "(3) Oscillatoria": "oscillatoria",
        "(4) Aphanizomenon": "aphanizomenon",
    }
    df = df.rename(columns=rename)

    df["site_order"] = df["site"].map(SITE_ORDER)
    df["month"] = df["sample_date"].dt.month
    df["day_of_year"] = df["sample_date"].dt.dayofyear
    df["season_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["season_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

    df["cells"] = df[TARGET_COL]
    df["log_cells"] = log1p10(df["cells"])
    df["alert"] = (df["cells"] >= 1000).astype(float)

    for col in ["microcystis", "anabaena", "oscillatoria", "aphanizomenon"]:
        if col in df.columns:
            df[f"{col}_ratio"] = safe_div(df[col].fillna(0), df["cells"].fillna(0))

    # Carlson-style TSI proxy
    df["tsi_chla"] = 9.81 * positive_ln(df["chla"]) + 30.6
    df["tsi_transparency"] = 60 - 14.41 * positive_ln(df["transparency"])
    df["tsi_proxy_mean"] = df[["tsi_chla", "tsi_transparency"]].mean(axis=1)
    df["eutrophic_flag"] = df["tsi_proxy_mean"].ge(50).astype(float)
    df["hypertrophic_flag"] = df["tsi_proxy_mean"].ge(70).astype(float)

    df["temp_light_growth_proxy"] = np.maximum(df["water_temp"] - 20, 0) * df["sunshine_obs"].fillna(0)

    sequence_cols = [
        "cells", "log_cells", "alert", "water_temp", "ph", "do", "transparency", "turbidity", "chla",
        "tsi_proxy_mean", "microcystis", "anabaena", "oscillatoria", "aphanizomenon",
    ]
    for col in sequence_cols:
        if col in df.columns:
            df[f"prev_{col}"] = df.groupby("site")[col].shift(1)
            df[f"delta_{col}"] = df[col] - df[f"prev_{col}"]

    df["prev_sample_date"] = df.groupby("site")["sample_date"].shift(1)
    df["time_delta_days"] = (df["sample_date"] - df["prev_sample_date"]).dt.days
    df["next_sample_date"] = df.groupby("site")["sample_date"].shift(-1)
    df["target_cells_next"] = df.groupby("site")["cells"].shift(-1)
    df["target_log_cells_next"] = log1p10(df["target_cells_next"])
    df["target_alert_next"] = (df["target_cells_next"] >= 1000).astype(float)
    df.loc[df["target_cells_next"].isna(), "target_alert_next"] = np.nan
    return df


def add_hydro_topology_features(df):
    df = df.sort_values(["site", "sample_date"]).copy()
    lookup = {site: group.sort_values("sample_date") for site, group in df.groupby("site")}
    records = []

    for _, row in df.iterrows():
        upstream = UPSTREAM.get(row["site"])
        values = {
            "upstream_log_cells_21d": np.nan,
            "upstream_days_since": np.nan,
            "upstream_alert_21d": np.nan,
            "graph_decay_signal": np.nan,
        }
        if upstream in lookup:
            hist = lookup[upstream]
            hist = hist[
                (hist["sample_date"] < row["sample_date"])
                & (hist["sample_date"] >= row["sample_date"] - pd.Timedelta(days=21))
            ]
            if not hist.empty:
                latest = hist.iloc[-1]
                days = int((row["sample_date"] - latest["sample_date"]).days)
                decay = math.exp(-days / 7)
                flow_weight = 1.0
                if "hydro_inflow_7d_sum" in row and "hydro_outflow_7d_sum" in row:
                    inflow = row.get("hydro_inflow_7d_sum")
                    outflow = row.get("hydro_outflow_7d_sum")
                    if pd.notna(inflow) and pd.notna(outflow):
                        flow_weight = float(np.tanh((inflow + 1) / (outflow + 1)) + 0.5)
                values = {
                    "upstream_log_cells_21d": latest["log_cells"],
                    "upstream_days_since": days,
                    "upstream_alert_21d": latest["alert"],
                    "graph_decay_signal": latest["log_cells"] * decay * flow_weight,
                }
        records.append(values)
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)

## 6. 모델 입력 테이블 생성

In [ ]:
quality_feat = add_quality_features(quality_raw)
hydro_feat = build_hydro_features(dam_raw, geum_dam)
weather_feat = build_weather_features(quality_feat[["site", "sample_date"]], kma_daily)

master = quality_feat.merge(hydro_feat, on="sample_date", how="left")
master = master.merge(weather_feat, on=["site", "sample_date"], how="left")
master = add_hydro_topology_features(master)

model_df = master.dropna(subset=["target_log_cells_next", "target_alert_next"]).copy()

print("master:", master.shape)
print("model_df:", model_df.shape)
master.head()

## 7. 학습/검증/테스트 분리

시계열 문제이므로 무작위 분할을 쓰지 않습니다.

- Train: 2016~2022
- Validation: 2023
- Test: 2024~2025

In [ ]:
train = model_df[model_df["sample_date"].dt.year <= 2022].copy()
valid = model_df[model_df["sample_date"].dt.year == 2023].copy()
test = model_df[model_df["sample_date"].dt.year >= 2024].copy()

len(train), len(valid), len(test)

In [ ]:
EXCLUDE_PREFIXES = ("target_", "next_")
EXCLUDE_COLUMNS = {
    "조사일", "채수위치", "sample_date", "prev_sample_date", "next_sample_date",
    "cells", "log_cells", "alert", "발령단계",
}


def select_features(df, train_df):
    candidates = []
    for col in df.columns:
        if col in EXCLUDE_COLUMNS or col.startswith(EXCLUDE_PREFIXES):
            continue
        if pd.api.types.is_numeric_dtype(df[col]) or col == "site":
            candidates.append(col)

    numeric = [col for col in candidates if pd.api.types.is_numeric_dtype(df[col])]
    categorical = [col for col in candidates if col not in numeric]

    # 학습 구간에서 전부 결측인 피처는 제외
    numeric = [col for col in numeric if train_df[col].notna().any()]
    categorical = [col for col in categorical if train_df[col].notna().any()]
    return numeric, categorical


numeric_features, categorical_features = select_features(model_df, train)
features = numeric_features + categorical_features

print("numeric:", len(numeric_features))
print("categorical:", categorical_features)
print("total features:", len(features))

## 8. 모델 학습

- 회귀: 다음 채수일 `log10(유해남조류 세포수 + 1)` 예측
- 분류: 다음 채수일 `관심 이상 여부` 예측

분류 threshold는 validation set에서 recall과 F1을 함께 고려해 선택합니다.

In [ ]:
def make_preprocessor(numeric, categorical):
    transformers = []
    if numeric:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric,
        ))
    if categorical:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ))
    return ColumnTransformer(transformers)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def choose_threshold(y_true, proba):
    best_score, best_threshold = -1, 0.5
    for threshold in np.linspace(0.1, 0.9, 33):
        pred = (proba >= threshold).astype(int)
        recall = recall_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)
        score = 0.7 * recall + 0.3 * f1
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold


preprocessor = make_preprocessor(numeric_features, categorical_features)

x_train = train[features]
x_valid = valid[features]
x_test = test[features]

y_reg_train = train["target_log_cells_next"]
y_reg_valid = valid["target_log_cells_next"]
y_reg_test = test["target_log_cells_next"]

y_cls_train = train["target_alert_next"].astype(int)
y_cls_valid = valid["target_alert_next"].astype(int)
y_cls_test = test["target_alert_next"].astype(int)

In [ ]:
# 회귀 모델
reg_model = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=300, learning_rate=0.04)),
])
reg_model.fit(x_train, y_reg_train)

valid_reg_pred = reg_model.predict(x_valid)
test_reg_pred = reg_model.predict(x_test)

reg_metrics = pd.DataFrame([
    {"task": "regression", "split": "valid", "model": "hist_gradient_boosting", "rmse_log": rmse(y_reg_valid, valid_reg_pred), "mae_log": mean_absolute_error(y_reg_valid, valid_reg_pred)},
    {"task": "regression", "split": "test", "model": "hist_gradient_boosting", "rmse_log": rmse(y_reg_test, test_reg_pred), "mae_log": mean_absolute_error(y_reg_test, test_reg_pred)},
])
reg_metrics

In [ ]:
# 분류 모델 2개 비교
classifiers = {
    "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE, max_iter=300, learning_rate=0.04),
    "random_forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        n_jobs=-1,
    ),
}

cls_metrics = []
cls_models = {}
cls_thresholds = {}

for name, estimator in classifiers.items():
    model = Pipeline([("prep", preprocessor), ("model", estimator)])
    model.fit(x_train, y_cls_train)
    valid_proba = model.predict_proba(x_valid)[:, 1]
    threshold = choose_threshold(y_cls_valid, valid_proba)
    cls_models[name] = model
    cls_thresholds[name] = threshold

    for split_name, x, y in [("valid", x_valid, y_cls_valid), ("test", x_test, y_cls_test)]:
        proba = model.predict_proba(x)[:, 1]
        pred = (proba >= threshold).astype(int)
        cls_metrics.append({
            "task": "classification",
            "split": split_name,
            "model": name,
            "threshold": threshold,
            "accuracy": accuracy_score(y, pred),
            "precision_alert": precision_score(y, pred, zero_division=0),
            "recall_alert": recall_score(y, pred, zero_division=0),
            "f1_alert": f1_score(y, pred, zero_division=0),
        })

cls_metrics = pd.DataFrame(cls_metrics)
cls_metrics

## 9. 결과 저장

실행 결과는 `outputs/model_pipeline/` 아래에 저장됩니다.

In [ ]:
metrics = pd.concat([reg_metrics, cls_metrics], ignore_index=True)

predictions = test[["site", "sample_date", "target_cells_next", "target_log_cells_next", "target_alert_next"]].copy()
predictions["pred_log_cells"] = test_reg_pred
predictions["pred_cells"] = np.power(10, predictions["pred_log_cells"]) - 1

for name, model in cls_models.items():
    proba = model.predict_proba(x_test)[:, 1]
    predictions[f"{name}_alert_proba"] = proba
    predictions[f"{name}_alert_pred"] = (proba >= cls_thresholds[name]).astype(int)

master.to_csv(OUTPUT_DIR / "tables/master_table.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"feature": features, "type": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features)}).to_csv(
    OUTPUT_DIR / "tables/feature_list.csv", index=False, encoding="utf-8-sig"
)
metrics.to_csv(OUTPUT_DIR / "tables/metrics.csv", index=False, encoding="utf-8-sig")
predictions.to_csv(OUTPUT_DIR / "tables/predictions.csv", index=False, encoding="utf-8-sig")

joblib.dump(reg_model, OUTPUT_DIR / "models/hist_gradient_boosting_regressor.joblib")
for name, model in cls_models.items():
    joblib.dump(model, OUTPUT_DIR / f"models/{name}_classifier.joblib")

print("saved to", OUTPUT_DIR)
metrics

## 10. 해석 포인트

- `rmse_log`는 세포수를 `log10(x + 1)`로 변환한 회귀 오차입니다.
- 조류경보 문제에서는 위험 미탐이 중요하므로 `recall_alert`를 특히 봅니다.
- TSI proxy, residence proxy, graph feature는 모델 성능뿐 아니라 보고서의 설명력에도 사용합니다.
- 장기 ASOS/AWS 기상자료가 추가되면 기상 rolling feature의 활용도가 커집니다.